In [ ]:
from pyspark.sql import functions as F

CATALOG = "spotify_etl"
SCHEMA = "bronze"
TABLE = "bronze_playlists"

dbutils.widgets.text("raw_base_path", "/Workspace/Users/pacioianu4@gmail.com/Files/spotify-end-to-end-api-project/data/raw", "RAW base path")
RAW_BASE_PATH = dbutils.widgets.get("raw_base_path").rstrip("/")


In [ ]:
import os, json

def collect():
    rows = []
    entity_path = f"{RAW_BASE_PATH}/me_playlists"
    if not os.path.exists(entity_path): return rows
    for root, dirs, files in os.walk(entity_path):
        for fn in files:
            if fn.startswith("page_") and fn.endswith(".json") and not fn.endswith("_meta.json"):
                with open(os.path.join(root, fn), "r", encoding="utf-8") as f:
                    data = json.load(f)
                for p in data.get("items", []):
                    pid = p.get("id")
                    if not pid: continue
                    fo = p.get("followers")
                    rows.append({
                        "playlist_id": pid, "playlist_name": p.get("name"),
                        "owner_name": (p.get("owner") or {}).get("display_name"),
                        "followers": fo.get("total") if isinstance(fo, dict) else None,
                        "total_tracks": (p.get("tracks") or {}).get("total"),
                        "description": p.get("description"), "snapshot_id": p.get("snapshot_id"),
                    })
    return rows

rows = collect()
if rows:
    df = spark.createDataFrame(rows).dropDuplicates(["playlist_id"])
    df = df.withColumn("processing_date", F.current_date())
    df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")
    print(f"Wrote {df.count()} rows to {CATALOG}.{SCHEMA}.{TABLE}")
else:
    print(f"No data for {TABLE}")
